# IndicOCR server on free Colab GPU for Bhu-Digitize

1. **Runtime → Change runtime type → T4 GPU**.
2. On https://huggingface.co/bodhan-ai/indic-ocr sign in and **accept the Indic Open Model License**.
3. Run the cells in order. The last cell prints a URL and token — put them in `backend\.env` on your laptop and restart Bhu-Digitize.

Keep this tab open while you use the app (Colab stops the server when the session ends).

In [ ]:
!nvidia-smi

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste a Hugging Face token (read access)

In [ ]:
# install IndicOCR exactly as its model card says
import subprocess
IDP = subprocess.check_output(['python','-c',"from huggingface_hub import snapshot_download as d; print(d('bodhan-ai/indic-ocr'))"]).decode().strip()
print(IDP)
!cd "$IDP" && bash ./install.sh
!pip -q install fastapi uvicorn python-multipart
# show where the indic_ocr package lives (should print a path)
!find "$IDP" -maxdepth 3 -type d -name indic_ocr
import os; os.environ["PYTHONPATH"] = IDP + ":" + os.path.join(IDP, "src") + ":" + os.environ.get("PYTHONPATH", "")

In [ ]:
%%writefile server.py
"""IndicOCR (Bodhan AI, open weights) as a small HTTP service for Bhu-Digitize.

Run this on a machine with an NVIDIA GPU (or a free Google Colab GPU — see colab_indic_ocr.ipynb):
    uvicorn server:app --host 0.0.0.0 --port 8000
Then in the app's backend/.env:
    OCR_ENGINE=indic_server
    INDIC_OCR_URL=http://<this-machine>:8000        (or the trycloudflare URL from Colab)
    INDIC_OCR_TOKEN=<same token as below>
"""
import os
import tempfile

import sys

from fastapi import FastAPI, File, Header, HTTPException, UploadFile
from huggingface_hub import snapshot_download

TOKEN = os.getenv("INDIC_OCR_TOKEN")    # shared secret; strongly recommended when exposed via a tunnel
REPO = snapshot_download(os.getenv("INDIC_OCR_REPO", "bodhan-ai/indic-ocr"))
# the `indic_ocr` package ships inside the model repo; install.sh installs its dependencies,
# so make the repo importable whether or not the package itself was pip-installed
for _p in (REPO, os.path.join(REPO, "src")):
    if (os.path.isdir(os.path.join(_p, "indic_ocr")) or os.path.isfile(os.path.join(_p, "indic_ocr.py"))) \
            and _p not in sys.path:
        sys.path.insert(0, _p)
from indic_ocr import IndicOCR  # noqa: E402
parser = IndicOCR.from_pretrained(REPO, table_format="markdown")
app = FastAPI(title="IndicOCR server")


@app.get("/health")
def health():
    return {"ok": True, "model": "bodhan-ai/indic-ocr"}


@app.post("/parse")
async def parse(file: UploadFile = File(...), x_token: str | None = Header(default=None)):
    if TOKEN and x_token != TOKEN:
        raise HTTPException(401, "bad token")
    data = await file.read()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as fh:
        fh.write(data)
        path = fh.name
    try:
        page = parser.parse(path)
    finally:
        os.unlink(path)
    return {"width": page.get("width"), "height": page.get("height"), "blocks": page.get("blocks", []),
            "markdown": page.get("markdown")}


In [ ]:
import secrets, os, subprocess, time
TOKEN = secrets.token_urlsafe(16)
os.environ['INDIC_OCR_TOKEN'] = TOKEN
subprocess.Popen('nohup uvicorn server:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &', shell=True, env=os.environ)
for _ in range(120):
    time.sleep(5)
    if 'Application startup complete' in open('server.log').read(): break
print(open('server.log').read()[-1500:])

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re
subprocess.Popen('nohup ./cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1 &', shell=True)
url=None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: url=m.group(0); break
print('Put these lines in backend\\.env on your laptop, then restart run_windows.bat:\n')
print('OCR_ENGINE=indic_server')
print('INDIC_OCR_URL=' + str(url))
print('INDIC_OCR_TOKEN=' + TOKEN)

In [ ]:
# quick self-test on a blank page
import requests, numpy as np, cv2
img = np.full((200,600),255,np.uint8); cv2.putText(img,'Khata 45',(20,120),cv2.FONT_HERSHEY_SIMPLEX,2,0,3)
r = requests.post('http://localhost:8000/parse', files={'file':('t.png', cv2.imencode('.png',img)[1].tobytes())}, headers={'x-token':TOKEN})
print(r.status_code, r.json().get('markdown'))